# Circumpolar boreal forest dynamics from ICESat-2 & Harmonized Landsat/Sentinel-2
+ AGBD v3.1 (30 m) for every available year, 2016-2025 (multi-yr ICESat-2 & HLS model; Duncanson et al. 2026 https://doi.org/10.1016/j.rse.2026.115406)
+ Forest stand age, 2020 (ensemble mean, 30 m)
+ Carbon accumulation, 2020 (ensemble mean, 30 m)
+ AGBD v3.1 OLS trends 2016-2025 and trend classes

Saved to `maps/Review_maps_v3.1_boreal_dynamics.html` with the shared Review map template (`lib/review_maps.py`).

In [ ]:
!pip install cogeo_mosaic==8.2.0

In [ ]:
import os
import sys

import geopandas

ICESAT2_BOREAL_LIB_PATH = '/projects/my-private-bucket/code/icesat2_boreal/lib'
sys.path.append(ICESAT2_BOREAL_LIB_PATH)
import maplib_folium
import mosaiclib

# Shared Review map page template (maap_tools/lib)
sys.path.append(os.path.abspath(os.path.join('..', 'lib')))
import review_maps

## AGBD v3.1 layers
One layer per year found in `mosaiclib.AGB_MOSAIC_JSON_FN_DICT`, sharing one colorbar. Note: zoom-in to display map tiles.

In [ ]:
AGB_YEARS = sorted(int(k.split('_')[0]) for k in mosaiclib.AGB_MOSAIC_JSON_FN_DICT if k.endswith('_v3.1_multiyr'))
print(AGB_YEARS)

AGB_TILES_LAYER_LIST = []
for i, YEAR in enumerate(AGB_YEARS):
    d = maplib_folium.make_tiles_layer_dict(
        None,
        mosaiclib.AGB_MOSAIC_JSON_FN_DICT[f'{YEAR}_v3.1_multiyr'],
        f"AGBD {YEAR} v3.1 [Mg/ha] (multi-yr ICESat-2 & HLS)",
        SHOW_CBAR=(i == 0),   # one shared colorbar for all AGBD layers
        PARAMS_DICT={"rescale": "0,125", "bidx": "1", "colormap_name": "viridis"},
    )
    d['caption'] = "AGBD v3.1 [Mg/ha]"
    AGB_TILES_LAYER_LIST.append(d)

## Forest stand age & carbon accumulation, 2020

In [ ]:
CACC_MOSAIC = mosaiclib.MISC_MOSAIC_JSON_FN_DICT['CACC_2020_v3.1_multiyr_nsims050']

AGE_LAYER = maplib_folium.make_tiles_layer_dict(
    None, CACC_MOSAIC,
    "Forest Stand Age 2020 [yr] (ensemble mean, 30m)",
    SHOW_CBAR=True,
    PARAMS_DICT={"rescale": "1,250", "bidx": "9", "colormap_name": 'gist_earth'},
)
CACC_LAYER = maplib_folium.make_tiles_layer_dict(
    None, CACC_MOSAIC,
    "Carbon Accumulation 2020 [MgC/ha/yr] (ensemble mean, 30m)",
    SHOW_CBAR=True,
    PARAMS_DICT={"rescale": "0,1", "bidx": "1", "colormap_name": 'turbo'},
)
AGE_CACC_LIST = [AGE_LAYER, CACC_LAYER]

## AGBD v3.1 trends, 2016-2025

In [ ]:
TREND_LAYER = maplib_folium.make_tiles_layer_dict(
    None,
    mosaiclib.MISC_MOSAIC_JSON_FN_DICT['TRENDOLS_AGB_v3.1_multiyr_2016-2025'],
    "AGBD v3.1 OLS trend 2016-2025 [Mg/ha/yr]",
    SHOW_CBAR=True,
    PARAMS_DICT={"rescale": "-2,2", "bidx": "1", "colormap_name": 'PiYG'},
)
TRENDCLASS_LAYER = maplib_folium.make_tiles_layer_dict(
    None,
    mosaiclib.MISC_MOSAIC_JSON_FN_DICT['TRENDCLASS_AGB_v3.1_multiyr_2016-2025'],
    "AGBD v3.1 trend classes 2016-2025",
    SHOW_CBAR=True,
    PARAMS_DICT={"rescale": "0,10", "bidx": "1", "colormap_name": 'RdBu_r'},
)
TRENDS_LIST = [TREND_LAYER, TRENDCLASS_LAYER]

# Map

In [ ]:
boreal_tiles = geopandas.read_file(mosaiclib.boreal_tile_index_path)

# Make ESRI gray the only basemap on at opening
for k, bm in maplib_folium.basemaps.items():
    bm.show = (k == 'basemap_gray')
    bm._children.pop(bm.get_name() + '_add', None)   # clear the 'add to map' step left by an earlier display

MAP_LAYERS = AGB_TILES_LAYER_LIST + AGE_CACC_LIST + TRENDS_LIST

m = maplib_folium.MAP_REGISTERED_DPS_RESULTS(
        boreal_tiles, None,
        CHECK_TILES_NAME='',
        map_width=750, map_height=500, SHOW_WIDGETS=False,
        ADD_TILELAYER = MAP_LAYERS,
    )

## Pixel query

In [ ]:
query_layers = [
    {'section': 'AGBD v3.1 [Mg/ha]', 'name': str(YEAR), 'point_url': review_maps.point_url(d, bidx=1), 'units': 'Mg/ha'}
    for YEAR, d in zip(AGB_YEARS, AGB_TILES_LAYER_LIST)
] + [
    {'section': 'Forest Stand Age 2020', 'name': 'mean', 'point_url': review_maps.point_url(AGE_LAYER, bidx=9), 'units': 'yr'},
    {'section': 'Forest Stand Age 2020', 'name': 'st dev', 'point_url': review_maps.point_url(AGE_LAYER, bidx=10), 'units': 'yr'},
    {'section': 'Carbon Accumulation 2020', 'name': 'mean', 'point_url': review_maps.point_url(CACC_LAYER, bidx=1), 'units': 'MgC/ha/yr'},
    {'section': 'Carbon Accumulation 2020', 'name': 'st dev', 'point_url': review_maps.point_url(CACC_LAYER, bidx=2), 'units': 'MgC/ha/yr'},
    {'section': 'AGBD v3.1 trend 2016-2025', 'name': 'slope (OLS)', 'point_url': review_maps.point_url(TREND_LAYER, bidx=1), 'units': 'Mg/ha/yr'},
    {'section': 'AGBD v3.1 trend 2016-2025', 'name': 'p-value (Mann-Kendall)', 'point_url': review_maps.point_url(TREND_LAYER, bidx=7), 'units': ''},
    {'section': 'AGBD v3.1 trend 2016-2025', 'name': 'class', 'point_url': review_maps.point_url(TRENDCLASS_LAYER, bidx=1), 'units': ''},
]

m = maplib_folium.add_pixel_query(m, query_layers, decimals=2)
m

### Save to html

In [ ]:
from review_maps import COMPLETE, WIP

DYNAMICS = {
    'title': 'Circumpolar boreal forest dynamics from ICESat-2 &amp; Harmonized Landsat/Sentinel-2',
    'subtitle': 'NASA Arctic/Boreal Vulnerability Experiment (ABoVE)',
    'logos': ['nasa', 'above', 'umd'],
    'note': 'Use the layer-control panel (top right) to toggle layers and basemaps; click anywhere on the map '
            'to query pixel values. Zoom in to display map tiles.',
    'refs': [review_maps.REF_DUNCANSON],
    'rows': [
        {'layer': f'AGBD v3.1, {AGB_YEARS[0]}&ndash;{AGB_YEARS[-1]}', 'resolution': '30&thinsp;m',
         'description': 'Annual aboveground biomass density (Mg/ha) from a multi-year model linking ICESat-2 ATL08 with '
                        'Harmonized Landsat/Sentinel-2 (HLS); one layer per year, shown from 0 to 125.',
         'citation': 'Duncanson et al. (2026) [1]', 'status': WIP},
        {'layer': 'Forest stand age, 2020', 'resolution': '30&thinsp;m',
         'description': 'Ensemble mean stand age (yr), shown from 1 to 250&thinsp;yr.',
         'citation': '&mdash;', 'status': WIP},
        {'layer': 'Carbon accumulation, 2020', 'resolution': '30&thinsp;m',
         'description': 'Ensemble mean woody carbon accumulation rate (MgC/ha/yr), shown from 0 to 1.',
         'citation': '&mdash;', 'status': WIP},
        {'layer': 'AGBD v3.1 OLS trend, 2016&ndash;2025', 'resolution': '30&thinsp;m',
         'description': 'Per-pixel OLS slope of annual v3.1 AGBD (Mg/ha/yr), shown from &minus;2 to +2. '
                        'Not yet complete across the circumpolar domain.',
         'citation': 'AGBD model: Duncanson et al. (2026) [1]', 'status': WIP},
        {'layer': 'AGBD v3.1 trend classes, 2016&ndash;2025', 'resolution': '30&thinsp;m',
         'description': 'Mann&ndash;Kendall trend classes (&alpha;&thinsp;=&thinsp;0.05): 1&ndash;4 significant '
                        'increases (strong to very weak), 5&ndash;6 non-significant (positive, negative), '
                        '7&ndash;10 significant decreases (very weak to strong).',
         'citation': 'AGBD model: Duncanson et al. (2026) [1]', 'status': WIP},
    ],
}

# Save next to the other published maps (repo's maps/ dir), relative to this notebook
out_html = os.path.abspath(os.path.join('..', 'maps', 'Review_maps_v3.1_boreal_dynamics.html'))
header_html = review_maps.build_header_html(DYNAMICS['title'], DYNAMICS['subtitle'], DYNAMICS['rows'], DYNAMICS['note'],
                                            DYNAMICS['refs'], logos=DYNAMICS['logos'])
review_maps.save_review_map(m, out_html, header_html)